# Hugging Face: models, tokenizers, and datasets

Hugging Face is where the open-model world lives. Nearly every open-weight model, from GPT-2 to Llama and Qwen, is published there, along with most public datasets. LM Studio and Ollama download from it under the hood, and so will your fine-tuning code. In this module you learn to use it from Python, then open up a real model and watch it predict the next token. By the end you'll know what's inside a model repo and exactly which token ids a model sees when you send it a chat.

In this notebook:

1. The pieces, and a tour in your browser
2. The Hub from Python
3. The cache
4. Tokenizers: how text becomes numbers
5. Chat templates: what the model actually sees
6. Running a model by hand, one token at a time
7. Pipelines
8. Datasets
9. Push your own dataset to the Hub (optional)

Before you start:

- Finish modules 01 and 02 first. This notebook builds on the chat messages, temperature, and KV cache from module 01 and the tensors and softmax from module 02.
- You need an internet connection. The notebook downloads about 2 GB into the Hugging Face cache: GPT-2, a 0.5B Qwen model, a small sentiment model, and a small dataset. Plan on 1.5 to 2 hours, plus download time.
- Section 9 is optional and needs your Hugging Face write token (`uv run hf auth login`, done during setup).
- Set the notebook's kernel to the `.venv` in `ML_Lab`, then run the install cell below once.

In [ ]:
# One-time step: adds transformers and datasets to the ML_Lab project.
# Safe to rerun. uv finds the ML_Lab project from this folder, and a rerun changes nothing.
!uv add transformers datasets

In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Device:", device)
_ = torch.manual_seed(0)

GPT2 = "openai-community/gpt2"          # 124M parameters, a base model from 2019
QWEN = "Qwen/Qwen2.5-0.5B-Instruct"     # 0.5B parameters, a modern instruct model

## 1. The pieces, and a tour in your browser

The Hub (huggingface.co) is GitHub for models and datasets. Every model is a git repository of files. A typical model repo contains these:

| File | What it is |
|---|---|
| `config.json` | The architecture: number of layers, hidden size, attention heads, vocabulary size |
| `model.safetensors` | The weights, often split into several shards for big models |
| `tokenizer.json` | The tokenizer's vocabulary and merge rules |
| `tokenizer_config.json` | Tokenizer settings, usually including the chat template |
| `generation_config.json` | Default sampling settings |
| `README.md` | The model card: who made it, training data, intended use, license |

Each library does one job:

| Library | Job |
|---|---|
| `huggingface_hub` | Download and upload files, manage your account and the local cache. Gives you the `hf` CLI |
| `transformers` | The model code. Turns `config.json` and the weights into a working PyTorch model |
| `tokenizers` | Fast tokenizers, written in Rust. `transformers` uses it automatically |
| `datasets` | Load, process, and stream datasets efficiently |

`mlx-lm` (module 07) reads the same repos. The `mlx-community` organization on the Hub publishes MLX conversions of popular models, so most models you'd want to run on your Mac already exist there in MLX format.

### Tour the Hub in your browser (about 10 minutes)

Do this before running any code. Seeing the files by hand makes everything below concrete.

1. Open [huggingface.co/Qwen/Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct). Read the model card and find the license.
2. Open the "Files and versions" tab. Click `config.json` and find `num_hidden_layers`, `hidden_size`, and `vocab_size`.
3. Find the model tree on the right side of the model page. It shows fine-tunes and quantizations other people built from this model. Click into a quantized one.
4. [Search for "Qwen2.5-1.5B-Instruct" under the `mlx-community` organization](https://huggingface.co/models?author=mlx-community&search=Qwen2.5-1.5B-Instruct), open the 4-bit version, and compare its files to the original.
5. Open a gated model like [meta-llama/Llama-3.2-1B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct). You'll see you have to accept a license before downloading. Accept it if you want to use Llama models later.

What to notice:

- Both repos carry the same tokenizer (`vocab.json`, `merges.txt`, `tokenizer.json`), so the same text becomes the same token ids. What changes is the weights and the config. The MLX `config.json` gains a `quantization` entry (4 bits, group size 64), and its `model.safetensors` is about 0.87 GB against 3.09 GB for the original, a little over a quarter. It's not exactly a quarter because every group of 64 weights also stores its own scale and offset.
- The license lives in the model card and in the repo's tags. You'll read the tag from Python in exercise 5.
- A model repo is just files. The next section inspects one from code without downloading the weights.

## 2. The Hub from Python

A model on the Hub is just a repo of files. `HfApi` lets you inspect one without downloading it.

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

api = HfApi()
info = api.model_info(GPT2, files_metadata=True)

print("Downloads (last 30 days):", info.downloads)
print("Likes:", info.likes)
print()
for f in info.siblings:
    size_mb = (f.size or 0) / 1e6
    if size_mb > 0.01:
        print(f"{f.rfilename:40} {size_mb:10.1f} MB")

The repo holds the same weights in several formats (`model.safetensors`, `pytorch_model.bin`, ONNX, and others). Libraries download only the file they need, and `safetensors` is the modern default because it loads fast and can't run hidden code the way old pickle-based `.bin` files can.

Now download one small file and read the architecture.

In [ ]:
config_path = hf_hub_download(GPT2, "config.json")
print("Saved to:", config_path)

with open(config_path) as f:
    config = json.load(f)

for key in ["n_layer", "n_head", "n_embd", "n_positions", "vocab_size"]:
    print(f"{key:12} {config[key]}")

Read those as: 12 transformer blocks, 12 attention heads per block, each token is represented by a vector of 768 numbers, the model can see at most 1024 tokens at once, and it knows 50,257 distinct tokens. In module 06 you'll build this exact architecture with much smaller numbers.

## 3. The cache

`hf_hub_download` saved that file under `~/.cache/huggingface/hub`. Every download from any library goes there, no matter which folder you run code from, so models are shared across projects and only downloaded once. This cell lists what's cached and how big each repo is:

In [ ]:
from huggingface_hub import scan_cache_dir, constants

import os

print("Cache folder:", constants.HF_HUB_CACHE)
if os.path.exists(constants.HF_HUB_CACHE):
    cache = scan_cache_dir()
    for repo in sorted(cache.repos, key=lambda r: r.size_on_disk, reverse=True):
        print(f"{repo.repo_type:8} {repo.repo_id:45} {repo.size_on_disk / 1e6:10.1f} MB")
else:
    print("(empty: nothing downloaded yet)")

The `hf` CLI that ships with `huggingface_hub` shows the same thing. Listing is read-only, so it's safe to run from here:

In [ ]:
!uv run hf cache ls

To delete one repo from the cache, run this in a terminal. It asks for confirmation before deleting, and `--dry-run` previews what would go:

```bash
uv run hf cache rm model/openai-community/gpt2 --dry-run   # preview
uv run hf cache rm model/openai-community/gpt2             # delete
```

The target is the repo type plus the repo id exactly as the cache listing shows it. Deleting is safe in the sense that the next `from_pretrained` call just downloads it again.

Run the scan cell above again at the end of the notebook to see what this module downloaded (about 2 GB in total).

## 4. Tokenizers

Models don't see characters or words. They see integers, one per token. A tokenizer splits text into pieces from a fixed vocabulary and maps each piece to its id.

In [ ]:
tok = AutoTokenizer.from_pretrained(GPT2)

text = "Darvis helps Virginia Tech students plan their courses."
ids = tok.encode(text)
tokens = tok.convert_ids_to_tokens(ids)

print(len(text.split()), "words ->", len(ids), "tokens")
for t, i in zip(tokens, ids):
    print(f"{i:6}  {t!r}")
print()
print("Decoded back:", tok.decode(ids))

The `Ġ` character marks a leading space. GPT-2's tokenizer treats " plan" and "plan" as different tokens. Common words get one token; rarer words like "Darvis" get split into pieces.

Predict how each of these will tokenize before you run the cell.

In [ ]:
def show(tokenizer, text):
    pieces = tokenizer.convert_ids_to_tokens(tokenizer.encode(text))
    print(f"{len(pieces):3} tokens | " + " | ".join(pieces))

for s in [
    "hello", " hello", "Hello", "HELLO",
    "strawberry",
    "12345 + 67890",
    "def add(a, b): return a + b",
    "¿Dónde está la biblioteca?",
    "こんにちは、元気ですか",
]:
    show(tok, s)

What to notice:

- Capitalization and leading spaces change the tokens. To the model, "hello" and " Hello" are unrelated ids.
- Numbers get chopped into arbitrary chunks. This is a big reason models are unreliable at arithmetic.
- "strawberry" is three tokens (`st`, `raw`, `berry`), not ten letters. The model never sees the individual r's, which is why counting letters is hard for it.
- Non-English text costs far more tokens per word. Japanese characters can take up to three tokens each (the 11-character sentence above became 14 tokens, because several characters took two). Since APIs bill per token and context windows are measured in tokens, the same content in some languages is more expensive and fits less.

Now compare GPT-2's 2019 tokenizer with a modern one.

In [ ]:
qtok = AutoTokenizer.from_pretrained(QWEN)
print("GPT-2 vocab size:", tok.vocab_size)
print("Qwen  vocab size:", len(qtok))
print()

samples = {
    "English": "Which professor for CS 3114 has the best grade distribution this semester?",
    "Code": "for i in range(10):\n    print(i * i)",
    "Japanese": "こんにちは、元気ですか。今日はいい天気ですね。",
}
for name, s in samples.items():
    print(f"{name:9} GPT-2: {len(tok.encode(s)):3} tokens   Qwen: {len(qtok.encode(s)):3} tokens")

A bigger vocabulary usually means fewer tokens for the same text (cheaper, faster, more fits in context), at the cost of a larger embedding table. Modern tokenizers also train on much more code and multilingual text, so Qwen does far better on the Japanese sample (12 tokens against 30) and a little better on the code.

The English sample goes the other way: Qwen needs 17 tokens against GPT-2's 14. The whole difference comes from "3114". GPT-2 chops " 3114" into two chunks, while Qwen splits every number into single digits and gives the leading space its own token, which makes five. One token per digit is a deliberate design choice that makes numbers more regular than GPT-2's arbitrary chunks. The price is a longer sequence whenever the text has numbers, and Darvis questions are full of course numbers, so keep this in mind for exercise 1.

Special tokens are reserved ids with structural meaning, like "end of text" or "start of a message".

In [ ]:
print("GPT-2 special tokens:", tok.all_special_tokens)
print("Qwen special tokens: ", qtok.all_special_tokens[:12])
print("Qwen end-of-sequence:", qtok.eos_token)

## 5. Chat templates

In module 01 you sent the model a list of messages with roles. The model doesn't receive a list. It receives one long string, built by the chat template stored in the tokenizer's config. Here's the exact text a Qwen model sees:

In [ ]:
messages = [
    {"role": "system", "content": "You are a concise academic advisor."},
    {"role": "user", "content": "Should I take CS 3114 and CS 3214 in the same semester?"},
]

prompt_text = qtok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt_text)

Each message is wrapped in `<|im_start|>role ... <|im_end|>` markers, and `add_generation_prompt=True` adds an open `<|im_start|>assistant` turn at the end, which tells the model it's its turn to talk. The model learned this exact format during instruction tuning. Llama, Gemma, and Mistral each use a different format, which is why the template ships with each tokenizer.

This matters a lot in module 07. When you fine-tune, your training examples get rendered through this template, and if training and inference use different formats, the model falls apart.

If you leave out the system message, Qwen's template inserts a default one. Try it:

In [ ]:
print(qtok.apply_chat_template(messages[1:], tokenize=False, add_generation_prompt=True))

## 6. Running a model by hand

Load GPT-2 and run one forward pass. The output is a score (logit) for every token in the vocabulary, at every position.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(GPT2).to(device)
model.eval()
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M   dtype: {model.dtype}")

inputs = tok("The best way to learn machine learning is", return_tensors="pt").to(device)
print("input_ids shape:", tuple(inputs["input_ids"].shape))

with torch.no_grad():
    logits = model(**inputs).logits
print("logits shape:   ", tuple(logits.shape), " = (batch, sequence length, vocab size)")

Before you run the next cell: which logits do we use to predict the next word, and why only those?

In [ ]:
next_logits = logits[0, -1]                      # the last position predicts what comes next
probs = torch.softmax(next_logits, dim=-1)       # the softmax you wrote in module 02
top = torch.topk(probs, 10)

for p, i in zip(top.values, top.indices):
    print(f"{p.item():6.1%}  {tok.decode(i)!r}")

That's the whole job of a language model: a probability for every possible next token. Everything else (chat, code, reasoning) is built by calling this repeatedly.

Now the temperature knob from module 01, made concrete. Dividing the logits by the temperature before the softmax sharpens or flattens the distribution.

In [ ]:
for T in [0.3, 1.0, 2.0]:
    p = torch.softmax(next_logits / T, dim=-1)
    entropy = -(p * torch.log(p + 1e-12)).sum().item()
    top3 = torch.topk(p, 3)
    shown = ", ".join(f"{tok.decode(i)!r} {v.item():.0%}" for v, i in zip(top3.values, top3.indices))
    print(f"T={T}: top token probability {top3.values[0].item():5.1%}  entropy {entropy:5.2f}  | {shown}")

Generation is a loop: predict, pick a token, append it, repeat. Here's greedy decoding (always take the top token), written out by hand.

In [ ]:
ids = inputs["input_ids"]
with torch.no_grad():
    for _ in range(25):
        next_id = model(ids).logits[:, -1].argmax(dim=-1, keepdim=True)
        ids = torch.cat([ids, next_id], dim=1)
print(tok.decode(ids[0]))

This loop is slow on purpose: every step reruns the whole sequence through the model. Real runtimes keep the keys and values from earlier positions in memory so each new token only needs one position of work. That's the KV cache you watched grow in module 01.

`model.generate` does the same loop, with caching and sampling options built in.

In [ ]:
out = model.generate(
    **inputs,
    max_new_tokens=40,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    pad_token_id=tok.eos_token_id,
)
print(tok.decode(out[0], skip_special_tokens=True))

GPT-2 is a base model. It was only trained to continue internet text, so it reads your question as the start of a document to continue. It may even give the right answer here, because "capital of France" is common in web text. Watch what it does after the first answer: a base model has no notion of a turn ending, so it keeps writing more of the page.

In [ ]:
q = tok("Q: What is the capital of France?\nA:", return_tensors="pt").to(device)
out = model.generate(**q, max_new_tokens=40, do_sample=False, pad_token_id=tok.eos_token_id)
print(tok.decode(out[0], skip_special_tokens=True))

Now an instruct model, prompted through its chat template. `return_dict=True` gives back both the token ids and the attention mask, ready to pass to `generate`.

In [ ]:
qmodel = AutoModelForCausalLM.from_pretrained(QWEN).to(device)
qmodel.eval()
print(f"Parameters: {sum(p.numel() for p in qmodel.parameters()) / 1e6:.0f}M   dtype: {qmodel.dtype}")

chat_inputs = qtok.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
).to(device)

with torch.no_grad():
    out = qmodel.generate(**chat_inputs, max_new_tokens=120, do_sample=False)

new_tokens = out[0, chat_inputs["input_ids"].shape[1]:]   # drop the prompt, keep only the reply
print(qtok.decode(new_tokens, skip_special_tokens=True))

Same mechanism as GPT-2 (predict the next token in a loop), but this model was trained further on conversations in that template, so it answers. The gap between base and instruct models is the gap that fine-tuning closes.

Free the memory before moving on.

In [ ]:
del qmodel
if device.type == "mps":
    torch.mps.empty_cache()

## 7. Pipelines

A pipeline bundles tokenizer, model, and post-processing into one call. Good for prototypes. Now you know what it hides.

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=device,
)
print(classifier([
    "This professor explains everything clearly.",
    "The exams had nothing to do with the lectures.",
    "It was fine I guess.",
]))

## 8. Datasets

The `datasets` library stores data in Apache Arrow, a columnar format that's memory-mapped from disk. You can work with datasets far larger than your RAM.

In [ ]:
from datasets import load_dataset

ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")
print(ds)

In [ ]:
train = ds["train"]
print(train.features)
print()
print("One row:", train[0])
print()
print("A slice is a dict of columns:", train[:2])
print()
label_names = train.features["label"].names
print("Label names:", label_names)

`map` applies a function to every row. With `batched=True` it processes many rows per call, which is much faster for tokenizers. The result is cached on disk, so rerunning is instant.

In [ ]:
def add_token_counts(batch):
    encoded = tok(batch["text"])
    return {"n_tokens": [len(ids) for ids in encoded["input_ids"]]}

train = train.map(add_token_counts, batched=True)
print(train)
print("Average tokens per review:", sum(train["n_tokens"]) / len(train))

long_reviews = train.filter(lambda row: row["n_tokens"] > 50)
print("Reviews over 50 tokens:", len(long_reviews))

df = train.to_pandas()
print(df.groupby("label")["n_tokens"].describe())

In [ ]:
# Splitting and sampling
small = train.shuffle(seed=0).select(range(1000))
splits = small.train_test_split(test_size=0.2, seed=0)
print(splits)

Streaming reads a dataset lazily over the network, so you can peek at data that's far too big to download. FineWeb-Edu is a filtered web crawl of the kind used to pretrain LLMs. The sample config alone is about 10 billion tokens.

In [ ]:
fineweb = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)

for row in fineweb.take(3):
    print(row["text"][:300].replace("\n", " "), "...")
    print("-" * 80)

## 9. Push your own dataset (optional)

Module 07 fine-tunes on chat-formatted examples like the ones below. Here you'll publish a tiny private dataset to your account to learn the workflow. This needs a token with write access (you created one during setup). Set `PUSH = True` to run it.

In [ ]:
from datasets import Dataset

rows = [
    {"messages": [
        {"role": "user", "content": "who's better for cs3114, Alvarez or Chen?"},
        {"role": "assistant", "content": '{"intent": "professor_compare", "courses": ["CS 3114"], "professors": ["Alvarez", "Chen"], "term": null}'},
    ]},
    {"messages": [
        {"role": "user", "content": "what do i need before taking systems"},
        {"role": "assistant", "content": '{"intent": "prerequisites", "courses": ["CS 3214"], "professors": [], "term": null}'},
    ]},
]
sample = Dataset.from_list(rows)
print(sample)

PUSH = False
if PUSH:
    username = api.whoami()["name"]
    repo_id = f"{username}/darvis-router-sample"
    sample.push_to_hub(repo_id, private=True)
    print("Pushed:", f"https://huggingface.co/datasets/{repo_id}")
    print(load_dataset(repo_id, split="train"))

## Wrap-up

Why this matters: when you build on open models, the Hub is your supply chain. You'll pick base models by reading model cards and licenses (some forbid commercial use, which matters for anything you ship), pull datasets from it, and publish your own fine-tuned models to it, privately at first. Knowing exactly what text and token ids reach the model is also what separates a fine-tune that works from one that quietly falls apart.

You're done when you can explain what each file in a model repo does, and you can turn a list of chat messages into the exact token ids a model sees.

Going further:

- Module 05 trains your own models and tracks them in W&B. Module 07 fine-tunes a Qwen model on chat-formatted data like the sample in section 9, rendered through the same chat template you inspected in section 5.
- The Hugging Face LLM course (huggingface.co/learn) covers `transformers` and `datasets` in more depth if you want a second pass.

---
# Exercises

**Exercise 1: Token economics.** Write `count_tokens(tokenizer, text)`. Darvis gets a question averaging 40 words, and every request also sends a 300-word system prompt and gets a 120-word answer back. Using the Qwen tokenizer on realistic sample text (write your own question, system prompt, and answer), estimate tokens per request. If a hosted API charges `IN_PRICE` dollars per million input tokens and `OUT_PRICE` per million output tokens, what do 100,000 requests a month cost? Look up a real price for a model you'd consider and plug it in.

**Exercise 2: Sampling from scratch.** Write `sample_next(logits, temperature=1.0, top_k=None)` that returns one token id: divide by temperature, keep only the top k logits if `top_k` is set (set the rest to `-inf`), softmax, then sample with `torch.multinomial`. Use it in a loop to generate 30 tokens from GPT-2 at temperature 0.3 and at 1.5.

**Exercise 3: Dataset statistics.** Using the Qwen tokenizer and `map`, compute the token length of every review in the rotten_tomatoes training set. Plot a histogram with matplotlib and print the mean and 95th percentile. Do positive or negative reviews run longer?

**Exercise 4: Count parameters from the config.** Using only the numbers in GPT-2's `config.json`, compute its parameter count by hand, then compare to `sum(p.numel() for p in model.parameters())`. Hints: token embeddings are `vocab_size x n_embd`, position embeddings `n_positions x n_embd`. Each block has a LayerNorm (weight and bias), attention with a combined query/key/value projection `n_embd -> 3*n_embd` and an output projection `n_embd -> n_embd`, another LayerNorm, and an MLP `n_embd -> 4*n_embd -> n_embd`. Every linear layer has a bias. There's one final LayerNorm, and the output layer reuses the token embedding matrix, so it adds nothing.

**Exercise 5: Explore a model you care about.** Pick a model you might fine-tune later (for example `Qwen/Qwen2.5-1.5B-Instruct`). Without downloading the weights, use `HfApi().model_info` and `hf_hub_download` on `config.json` to find its layer count, hidden size, vocab size, context length, total weight size in GB, and license tag. Would it fit in your Mac's memory at 16-bit? At 4-bit?

---
# Solutions

In [ ]:
# Exercise 1
def count_tokens(tokenizer, text):
    return len(tokenizer.encode(text))

question = "Which professor for CS 3114 gives the best grades, and is the workload manageable alongside CS 3214 and a part-time job? I care more about learning than an easy A but I need to keep my GPA up for internships."
system_prompt = " ".join(["You are Darvis, an academic assistant for Virginia Tech students. Answer using the course data provided."] * 15)
answer = " ".join(["Based on recent grade distributions, Professor B has a higher average GPA and A rate for CS 3114."] * 6)

n_in = count_tokens(qtok, system_prompt) + count_tokens(qtok, question)
n_out = count_tokens(qtok, answer)
print(f"Input tokens per request:  {n_in}")
print(f"Output tokens per request: {n_out}")

IN_PRICE, OUT_PRICE = 0.50, 1.50   # placeholder $ per 1M tokens. Replace with a real model's prices.
requests = 100_000
monthly = requests * (n_in * IN_PRICE + n_out * OUT_PRICE) / 1e6
print(f"Monthly cost for {requests:,} requests: ${monthly:,.2f}")
print(f"Share of input cost from the system prompt: {count_tokens(qtok, system_prompt) / n_in:.0%}")

In [ ]:
# Exercise 2
def sample_next(logits, temperature=1.0, top_k=None):
    logits = logits / temperature
    if top_k is not None:
        kth_value = torch.topk(logits, top_k).values[..., -1, None]
        logits = logits.masked_fill(logits < kth_value, float("-inf"))
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)

for T in [0.3, 1.5]:
    ids = tok("The best way to learn machine learning is", return_tensors="pt")["input_ids"].to(device)
    with torch.no_grad():
        for _ in range(30):
            next_id = sample_next(model(ids).logits[:, -1], temperature=T, top_k=50)
            ids = torch.cat([ids, next_id], dim=1)
    print(f"T={T}:", tok.decode(ids[0]))
    print()

In [ ]:
# Exercise 3
import numpy as np
import matplotlib.pyplot as plt

lengths = ds["train"].map(lambda b: {"q_tokens": [len(x) for x in qtok(b["text"])["input_ids"]]}, batched=True)
q = np.array(lengths["q_tokens"])
labels = np.array(lengths["label"])

print(f"mean {q.mean():.1f}   95th percentile {np.percentile(q, 95):.0f}")
for i, name in enumerate(label_names):
    print(f"{name}: mean {q[labels == i].mean():.1f} tokens")

plt.hist(q, bins=40)
plt.xlabel("tokens per review"); plt.ylabel("count"); plt.title("rotten_tomatoes review lengths (Qwen tokenizer)")
plt.show()

In [ ]:
# Exercise 4
V, P, d, L = config["vocab_size"], config["n_positions"], config["n_embd"], config["n_layer"]

embeddings = V * d + P * d
layernorm = 2 * d
attention = (d * 3 * d + 3 * d) + (d * d + d)
mlp = (d * 4 * d + 4 * d) + (4 * d * d + d)
per_block = 2 * layernorm + attention + mlp
total = embeddings + L * per_block + layernorm

print(f"by hand:    {total:,}")
print(f"from model: {sum(p.numel() for p in model.parameters()):,}")
print(f"embeddings are {embeddings / total:.0%} of GPT-2 small")

In [ ]:
# Exercise 5
repo = "Qwen/Qwen2.5-1.5B-Instruct"
info5 = api.model_info(repo, files_metadata=True)
with open(hf_hub_download(repo, "config.json")) as f:
    c = json.load(f)

weight_gb = sum((s.size or 0) for s in info5.siblings if s.rfilename.endswith(".safetensors")) / 1e9
license_tags = [t for t in (info5.tags or []) if t.startswith("license:")]
print(f"layers {c['num_hidden_layers']}, hidden {c['hidden_size']}, vocab {c['vocab_size']}, context {c['max_position_embeddings']}")
print(f"weights on disk: {weight_gb:.1f} GB   license: {license_tags}")

params_b = weight_gb / 2   # stored in 16-bit, so 2 bytes per parameter
print(f"~{params_b:.1f}B params -> 16-bit ~{params_b * 2:.1f} GB, 4-bit ~{params_b * 0.5 + 0.3:.1f} GB. Your GPU budget is about 17.8 GB.")